In [2]:
import os
import time
import uuid
import pandas as pd
import fitz  # PyMuPDF
from typing import Optional
from pydantic import BaseModel, Field
from docling.document_converter import DocumentConverter

# Enable async execution in Jupyter (Needed if you use LlamaParse)
import nest_asyncio
nest_asyncio.apply()
from llama_parse import LlamaParse

class NCCNChunkMetadata(BaseModel):
    source: str = Field(..., description="File path or URL of the guideline")
    page: int = Field(..., description="Page number of the chunk")
    document_type: str = Field(default="Clinical Guideline", description="Type of document")
    publication_date: str = Field(..., description="Year of publication")
    section: Optional[str] = Field(None, description="Guideline section/heading")
    chunk_id: str = Field(default_factory=lambda: str(uuid.uuid4()), description="Unique identifier")

class DocumentChunk(BaseModel):
    text: str
    metadata: NCCNChunkMetadata

/tmp/ipykernel_64689/3832774614.py:13: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


In [3]:
class AdvancedIngestionPipeline:
    def __init__(self):
        # Initialize Docling for local structural and tabular parsing
        self.docling_converter = DocumentConverter()
        
        # Initialize LlamaParse requesting strict markdown output
        self.llama_parser = LlamaParse(
            result_type="markdown",
            verbose=True
        )

    def process_pymupdf(self, file_path: str, year: str) -> dict:
        """Baseline extraction: Fast, but loses complex table relationships."""
        start_time = time.time()
        doc = fitz.open(file_path)
        chunks = []
        
        for page_num, page in enumerate(doc):
            text = page.get_text("text").strip()
            if text:
                meta = NCCNChunkMetadata(
                    source=file_path, 
                    page=page_num + 1,
                    publication_date=year
                )
                chunks.append(DocumentChunk(text=text, metadata=meta))
                
        return {
            "method": "PyMuPDF", 
            "time_seconds": time.time() - start_time, 
            "chunks": chunks
        }

    def process_docling(self, file_path: str, year: str) -> dict:
        """Advanced local extraction: Preserves reading order and Markdown tables."""
        start_time = time.time()
        result = self.docling_converter.convert(file_path)
        chunks = []
        
        # Extract Standard Text
        for item in result.document.texts:
            meta = NCCNChunkMetadata(
                source=file_path, 
                page=item.prov[0].page_no if item.prov else 0,
                publication_date=year, 
                section="Standard Text"
            )
            chunks.append(DocumentChunk(text=item.text, metadata=meta))
            
        # Extract Tables as Markdown (Crucial for clinical guidelines)
        for table in result.document.tables:
            meta = NCCNChunkMetadata(
                source=file_path, 
                page=table.prov[0].page_no if table.prov else 0,
                publication_date=year, 
                section="Tabular Data"
            )
            # doc=result.document is passed to avoid deprecation warnings
            chunks.append(DocumentChunk(text=table.export_to_markdown(doc=result.document), metadata=meta))
            
        return {
            "method": "Docling", 
            "time_seconds": time.time() - start_time, 
            "chunks": chunks
        }

    def process_llamaparse(self, file_path: str, year: str) -> dict:
        """API-based extraction: Premium structural and table parsing via cloud vision models."""
        if "LLAMA_CLOUD_API_KEY" not in os.environ:
            print("Skipping LlamaParse: API Key not set in environment variables.")
            return {"method": "LlamaParse", "time_seconds": 0, "chunks": []}

        start_time = time.time()
        
        documents = self.llama_parser.load_data(file_path)
        chunks = []
        
        for doc in documents:
            blocks = doc.text.split('\n\n')
            for block in blocks:
                if block.strip():
                    meta = NCCNChunkMetadata(
                        source=file_path, 
                        page=0, 
                        publication_date=year, 
                        section="LlamaParse Block"
                    )
                    chunks.append(DocumentChunk(text=block.strip(), metadata=meta))
                    
        return {
            "method": "LlamaParse", 
            "time_seconds": time.time() - start_time, 
            "chunks": chunks
        }

In [7]:
# Set your LlamaCloud API Key if testing LlamaParse
os.environ["LLAMA_CLOUD_API_KEY"] = "llx-GTFHiS8J2S5fceKJDS6VPFfOmHcnopNtsZFkM4QBI7yoO16O"

TEST_FILE = "NCCN_Guidelines_dataset/guidelines/2023genetics_bop_3.2023_active_021323.pdf" 
YEAR = "2023"

pipeline = AdvancedIngestionPipeline()

print(f"Running PyMuPDF...")
pymupdf_results = pipeline.process_pymupdf(TEST_FILE, YEAR)

print("Running Docling (Local Document Parsing)...")
docling_results = pipeline.process_docling(TEST_FILE, YEAR)

print("Running LlamaParse (Cloud Vision API)...")
llamaparse_results = pipeline.process_llamaparse(TEST_FILE, YEAR)

# Generate Benchmark Report
df_report = pd.DataFrame([
    {"Method": "PyMuPDF", "Time (s)": round(pymupdf_results["time_seconds"], 2), "Chunks": len(pymupdf_results["chunks"])},
    {"Method": "Docling", "Time (s)": round(docling_results["time_seconds"], 2), "Chunks": len(docling_results["chunks"])},
    {"Method": "LlamaParse", "Time (s)": round(llamaparse_results["time_seconds"], 2), "Chunks": len(llamaparse_results["chunks"])}
])

display(df_report)

# Inspect output to verify table structures
print("\n--- SAMPLE DOCLING EXTRACTION ---")
if docling_results["chunks"]:
    table_chunks = [c for c in docling_results["chunks"] if c.metadata.section == "Tabular Data"]
    if table_chunks:
        print(table_chunks[0].text)

Running PyMuPDF...


[INFO] 2026-06-22 22:22:53,800 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-22 22:22:53,801 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-06-22 22:22:53,824 [RapidOCR] download_file.py:60: File exists and is valid: /home/creo/coding/Internship/Oncology_Research_Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-22 22:22:53,824 [RapidOCR] main.py:50: Using /home/creo/coding/Internship/Oncology_Research_Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-22 22:22:53,941 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-22 22:22:53,942 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-06-22 22:22:53,949 [RapidOCR] download_file.py:60: File exists and is valid: /home/creo/coding/Internship/Oncology_Research_Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2

Running Docling (Local Document Parsing)...


[INFO] 2026-06-22 22:22:54,002 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-22 22:22:54,003 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-06-22 22:22:54,050 [RapidOCR] download_file.py:60: File exists and is valid: /home/creo/coding/Internship/Oncology_Research_Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.pth
[INFO] 2026-06-22 22:22:54,050 [RapidOCR] main.py:50: Using /home/creo/coding/Internship/Oncology_Research_Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.pth


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Running LlamaParse (Cloud Vision API)...
Started parsing the file under job_id 5db005f6-9bde-4b36-a4f6-7ff76df5f8ad


,Method,Time (s),Chunks
0,PyMuPDF,0.25,148
1,Docling,29.20,2906
2,LlamaParse,23.42,1883



--- SAMPLE DOCLING EXTRACTION ---
| Gene   | Breast Cancer Risk and Management (First primary)                                                                                                                                                                                                  | Epithelial Ovarian Cancer Risk and Management                                                                                                                                                                                                      | Pancreatic Cancer Risk and Management 13-22 and Other Cancer Risks                                                                                                                                                                                                                      |
|--------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [8]:
import json

# Export the Docling chunks to a local JSON file
output_path = "nccn_parsed_corpus.json"

# Convert Pydantic models to standard dictionaries for JSON saving
serialized_chunks = [chunk.model_dump() for chunk in docling_results["chunks"]]

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(serialized_chunks, f, indent=2)

print(f"Successfully saved {len(serialized_chunks)} chunks to {output_path}")

Successfully saved 2906 chunks to nccn_parsed_corpus.json


In [19]:
type(serialized_chunks)

list

In [27]:
print(serialized_chunks[0])

{'text': 'NCCN Clinical Practice Guidelines in Oncology (NCCN Guidelines ® )', 'metadata': {'source': 'NCCN_Guidelines_dataset/guidelines/2023genetics_bop_3.2023_active_021323.pdf', 'page': 1, 'document_type': 'Clinical Guideline', 'publication_date': '2023', 'section': 'Standard Text', 'chunk_id': '2a57a171-4b15-4b2c-a962-2e4ab4626406'}}


In [28]:
for chunk in serialized_chunks[:10]:
    print(chunk['text'])
    print("=" * 50)
    print(chunk['metadata'])
    print("\n")

NCCN Clinical Practice Guidelines in Oncology (NCCN Guidelines ® )
{'source': 'NCCN_Guidelines_dataset/guidelines/2023genetics_bop_3.2023_active_021323.pdf', 'page': 1, 'document_type': 'Clinical Guideline', 'publication_date': '2023', 'section': 'Standard Text', 'chunk_id': '2a57a171-4b15-4b2c-a962-2e4ab4626406'}


Genetic/Familial High-Risk Assessment: Breast, Ovarian, and Pancreatic
{'source': 'NCCN_Guidelines_dataset/guidelines/2023genetics_bop_3.2023_active_021323.pdf', 'page': 1, 'document_type': 'Clinical Guideline', 'publication_date': '2023', 'section': 'Standard Text', 'chunk_id': 'db125ef8-bb24-46f4-a378-8555ef42107a'}


Version 3.2023 - February 13, 2023
{'source': 'NCCN_Guidelines_dataset/guidelines/2023genetics_bop_3.2023_active_021323.pdf', 'page': 1, 'document_type': 'Clinical Guideline', 'publication_date': '2023', 'section': 'Standard Text', 'chunk_id': 'ff9449cd-0697-4fcb-aa2d-fe6dd20389e8'}


NCCN.org
{'source': 'NCCN_Guidelines_dataset/guidelines/2023genetics_bop_3

In [29]:
class AdvancedIngestionPipeline:
    def __init__(self):
        # Initialize Docling for local structural and tabular parsing
        self.docling_converter = DocumentConverter()
        
        # Initialize LlamaParse requesting strict markdown output
        self.llama_parser = LlamaParse(
            result_type="markdown",
            verbose=True
        )

    def process_pymupdf(self, file_path: str, year: str) -> dict:
        """Baseline extraction: Fast, but loses complex table relationships."""
        start_time = time.time()
        doc = fitz.open(file_path)
        chunks = []
        
        for page_num, page in enumerate(doc):
            text = page.get_text("text").strip()
            if text:
                meta = NCCNChunkMetadata(
                    source=file_path, 
                    page=page_num + 1,
                    publication_date=year
                )
                chunks.append(DocumentChunk(text=text, metadata=meta))
                
        return {
            "method": "PyMuPDF", 
            "time_seconds": time.time() - start_time, 
            "chunks": chunks
        }

    def process_docling(self, file_path: str, year: str, target_chunk_size: int = 800, overlap: int = 150) -> dict:
        """Advanced local extraction: Aggregates tiny blocks, applies overlap, filters noise."""
        start_time = time.time()
        result = self.docling_converter.convert(file_path)
        
        raw_chunks = []
        
        # Extract Tables as Markdown 
        for table in result.document.tables:
            meta = NCCNChunkMetadata(
                source=file_path, 
                page=table.prov[0].page_no if table.prov else 0,
                publication_date=year, 
                section="Tabular Data"
            )
            raw_chunks.append(DocumentChunk(
                text=table.export_to_markdown(doc=result.document), 
                metadata=meta
            ))

        # Extract and Aggregate Standard Text
        current_text_buffer = []
        current_char_count = 0
        current_page = 1

        noise_filters = ["nccn.org", "continue", "table of contents", "discussion", "index", "all rights reserved"]

        for item in result.document.texts:
            text_snippet = item.text.strip()
            if not text_snippet:
                continue
                
            if any(noise in text_snippet.lower() for noise in noise_filters):
                continue

            item_page = item.prov[0].page_no if item.prov else current_page

            if current_char_count >= target_chunk_size or (item_page != current_page and current_text_buffer):
                combined_text = " ".join(current_text_buffer)
                
                if len(combined_text) > 120:
                    meta = NCCNChunkMetadata(
                        source=file_path, 
                        page=current_page,
                        publication_date=year, 
                        section="Aggregated Clinical Prose"
                    )
                    raw_chunks.append(DocumentChunk(text=combined_text, metadata=meta))

                words = combined_text.split()
                overlap_words = words[-max(1, int(len(words)*0.15)):] 
                current_text_buffer = [" ".join(overlap_words)] if overlap_words else []
                current_char_count = len(current_text_buffer[0]) if current_text_buffer else 0

            current_text_buffer.append(text_snippet)
            current_char_count += len(text_snippet) + 1
            current_page = item_page

        if current_text_buffer:
            combined_text = " ".join(current_text_buffer)
            if len(combined_text) > 120:
                meta = NCCNChunkMetadata(source=file_path, page=current_page, publication_date=year, section="Aggregated Clinical Prose")
                raw_chunks.append(DocumentChunk(text=combined_text, metadata=meta))

        return {
            "method": "Docling", 
            "time_seconds": time.time() - start_time, 
            "chunks": raw_chunks
        }

    def process_llamaparse(self, file_path: str, year: str) -> dict:
        """API-based extraction: Premium structural and table parsing via cloud vision models."""
        if "LLAMA_CLOUD_API_KEY" not in os.environ:
            print("Skipping LlamaParse: API Key not set in environment variables.")
            return {"method": "LlamaParse", "time_seconds": 0, "chunks": []}

        start_time = time.time()
        
        documents = self.llama_parser.load_data(file_path)
        chunks = []
        
        for doc in documents:
            blocks = doc.text.split('\n\n')
            for block in blocks:
                if block.strip():
                    meta = NCCNChunkMetadata(
                        source=file_path, 
                        page=0, 
                        publication_date=year, 
                        section="LlamaParse Block"
                    )
                    chunks.append(DocumentChunk(text=block.strip(), metadata=meta))
                    
        return {
            "method": "LlamaParse", 
            "time_seconds": time.time() - start_time, 
            "chunks": chunks
        }

In [30]:
# Set your LlamaCloud API Key if testing LlamaParse
os.environ["LLAMA_CLOUD_API_KEY"] = "llx-GTFHiS8J2S5fceKJDS6VPFfOmHcnopNtsZFkM4QBI7yoO16O"

TEST_FILE = "NCCN_Guidelines_dataset/guidelines/2023genetics_bop_3.2023_active_021323.pdf" 
YEAR = "2023"

pipeline = AdvancedIngestionPipeline()

print(f"Running PyMuPDF...")
pymupdf_results = pipeline.process_pymupdf(TEST_FILE, YEAR)

print("Running Docling (Local Document Parsing)...")
docling_results = pipeline.process_docling(TEST_FILE, YEAR)

print("Running LlamaParse (Cloud Vision API)...")
llamaparse_results = pipeline.process_llamaparse(TEST_FILE, YEAR)

# Generate Benchmark Report
df_report = pd.DataFrame([
    {"Method": "PyMuPDF", "Time (s)": round(pymupdf_results["time_seconds"], 2), "Chunks": len(pymupdf_results["chunks"])},
    {"Method": "Docling", "Time (s)": round(docling_results["time_seconds"], 2), "Chunks": len(docling_results["chunks"])},
    {"Method": "LlamaParse", "Time (s)": round(llamaparse_results["time_seconds"], 2), "Chunks": len(llamaparse_results["chunks"])}
])

display(df_report)

# Inspect output to verify table structures
print("\n--- SAMPLE DOCLING EXTRACTION ---")
if docling_results["chunks"]:
    table_chunks = [c for c in docling_results["chunks"] if c.metadata.section == "Tabular Data"]
    if table_chunks:
        print(table_chunks[0].text)

Running PyMuPDF...
Running Docling (Local Document Parsing)...


[INFO] 2026-06-22 22:57:48,907 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-22 22:57:48,908 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-06-22 22:57:48,931 [RapidOCR] download_file.py:60: File exists and is valid: /home/creo/coding/Internship/Oncology_Research_Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-22 22:57:48,931 [RapidOCR] main.py:50: Using /home/creo/coding/Internship/Oncology_Research_Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-06-22 22:57:49,059 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-06-22 22:57:49,059 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-06-22 22:57:49,062 [RapidOCR] download_file.py:60: File exists and is valid: /home/creo/coding/Internship/Oncology_Research_Assistant/.venv/lib/python3.12/site-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

Running LlamaParse (Cloud Vision API)...
Started parsing the file under job_id 5db005f6-9bde-4b36-a4f6-7ff76df5f8ad


,Method,Time (s),Chunks
0,PyMuPDF,0.26,148
1,Docling,31.90,622
2,LlamaParse,10.31,1883



--- SAMPLE DOCLING EXTRACTION ---
| Gene   | Breast Cancer Risk and Management (First primary)                                                                                                                                                                                                  | Epithelial Ovarian Cancer Risk and Management                                                                                                                                                                                                      | Pancreatic Cancer Risk and Management 13-22 and Other Cancer Risks                                                                                                                                                                                                                      |
|--------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [31]:
import json

# Export the Docling chunks to a local JSON file
output_path = "nccn_parsed_corpus.json"

# Convert Pydantic models to standard dictionaries for JSON saving
serialized_chunks = [chunk.model_dump() for chunk in docling_results["chunks"]]

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(serialized_chunks, f, indent=2)

print(f"Successfully saved {len(serialized_chunks)} chunks to {output_path}")

Successfully saved 622 chunks to nccn_parsed_corpus.json


In [32]:
for chunk in serialized_chunks[:10]:
    print(chunk['text'])
    print("=" * 50)
    print(chunk['metadata'])
    print("\n")

| Gene   | Breast Cancer Risk and Management (First primary)                                                                                                                                                                                                  | Epithelial Ovarian Cancer Risk and Management                                                                                                                                                                                                      | Pancreatic Cancer Risk and Management 13-22 and Other Cancer Risks                                                                                                                                                                                                                      |
|--------|---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------